# Model 1 — Calibration against official Israeli statistics (CBS)  (experiments only)

**Decision taken after notebook 01:** instead of adding Dataset 6 (US Craigslist), investigate calibrating the 7+8+3 regression with **official CBS data**.

**Rules of this notebook**
* The CBS numbers were read **directly from the official publication** (see `ml/src/cbs_reference.py` for table/page references) — not from press coverage.
* Calibration fixes the **level of a population mean**. It is *not* evidence about the error of any individual vehicle, and nothing here is presented as such.
* Nothing is exported or integrated.

**Questions**
1. What do the official CBS tables say, and are my transcriptions internally consistent?
2. How far is the uncalibrated model from them?
3. How would the calibration work, and does it hold **out of sample** against CBS marginals it was not fitted to?
4. Does the *procedure* really improve **individual-level** error? (tested where individual ground truth exists: transfer between UK / India / US markets)
5. What it cannot do.

In [1]:
import os, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.options.display.width = 170
pd.options.display.max_columns = 40
pd.options.display.float_format = "{:,.3f}".format
sys.path.insert(0, str(Path.cwd().parent / "src"))
from config import INTERIM_DIR
import cbs_reference as cbs
import calibration as cal_
import mileage_data as md_
import mileage_experiments as mx
from israel_registry import normalise_registry
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
try:
    display
except NameError:
    display = print
SMOKE = os.environ.get("SMARTCAR_SMOKE") == "1"

## 1. Official data (primary source) and consistency checks

In [2]:
age = pd.DataFrame(cbs.CBS_AGE_2022, index=["avg annual km (thousand)", "total km (million)", "vehicles"]).T
age["2021"] = pd.Series(cbs.CBS_AGE_2021); age["2020"] = pd.Series(cbs.CBS_AGE_2020)
display(age)
tot_km, tot_veh = age["total km (million)"].sum(), age["vehicles"].sum()
print(f"Check: bands sum to {tot_km:,.0f} M km and {tot_veh:,.0f} vehicles; published totals {cbs.CBS_PRIVATE_CAR_TOTAL_2022[1]:,} / {cbs.CBS_PRIVATE_CAR_TOTAL_2022[2]:,}")
print(f"Check: weighted mean of the bands = {tot_km*1e6/tot_veh/1000:.2f} (published {cbs.CBS_PRIVATE_CAR_TOTAL_2022[0]})")
print()
print("Private cars by ownership (Table 8, thousand km/yr):"); display(pd.DataFrame(cbs.CBS_OWNERSHIP_2022).T)
print("Private cars by propulsion (2022):", cbs.CBS_FUEL_2022, "| electric: NOT published by CBS (footnote 4)")
print("Fleet mean, private cars:", cbs.CBS_FLEET_PRIVATE)

,avg annual km (thousand),total km (million),vehicles,2021,2020
0-3,19.600,"17,286.000","881,923.000",19.300,17.500
4-6,16.000,"12,455.000","780,053.000",16.200,13.600
7-9,14.200,"8,900.000","627,158.000",14.400,12.200
10-13,12.600,"7,973.000","633,532.000",12.600,11.100
14-19,11.500,"4,993.000","432,820.000",11.200,10.100
20-29,9.300,548.000,"58,701.000",9.100,8.700
30+,4.600,51.000,"11,121.000",4.500,4.900


Check: bands sum to 52,206 M km and 3,425,308 vehicles; published totals 52,205 / 3,425,309
Check: weighted mean of the bands = 15.24 (published 15.2)

Private cars by ownership (Table 8, thousand km/yr):


,new,old,all
private,16.000,13.400,13.900
leasing,28.400,26.200,28.300
company,26.500,22.300,24.400
rental,26.500,23.200,26.100
total,19.600,13.700,15.200


Private cars by propulsion (2022): {'gasoline': 14.0, 'diesel': 22.8, 'hybrid': 21.1, 'gas_lpg': 18.8} | electric: NOT published by CBS (footnote 4)
Fleet mean, private cars: {2019: 16.2, 2020: 13.5, 2021: 15.3, 2022: 15.2, 2023: 14.7, 2024: 14.7}


**What the primary source adds (and corrects) compared with press coverage**
* The CBS measures **odometer readings recorded at the annual vehicle inspection, for the whole fleet** — Israel *has* real odometer data for every car; it is just not published per vehicle.
* **Ownership matters enormously:** leasing 28.3, company 24.4, rental 26.1 vs privately owned **13.9** thousand km/yr (Table 8). The registry has this category (`baalut`) for every plate; none of the training datasets has it.
  (This is *not* the same thing as `Owner_Type` = number of previous owners in Dataset 8.)
* Press figures such as "16,700 km for cars up to 3 years" refer to *privately owned* new cars; the all-ownership figure is **19.6** (Table 4). The fleet mean fell from 15.2 (2022) to **14.7 (2023, 2024)**.
* Electric vehicles are **not** published by the CBS yet (too new to have passed the inspection test).

## 2. Expected odometer implied by the official yearly rates alone
If yearly driving by age were stationary, a car of age *A* would have driven `Σ rate(age band of a)` for `a = 1..A`. Assumptions (all approximate): stationarity across cohorts, first-year partial use, survivorship (old cars that survive may have driven less).
Compare with the median odometer of the used-car listings in each training market.

In [3]:
clean = {k: md_.load_clean(k)[0] for k in md_.LOADERS}
if SMOKE:
    clean = {k: (v.sample(20000, random_state=0) if len(v) > 20000 else v) for k, v in clean.items()}
IN = pd.concat([clean["d8_in"], clean["d3_cardekho"]], ignore_index=True)
MARKETS = {"UK": clean["d7_uk"], "India": IN, "US": clean["d6_craigslist"]}

rates22 = {b: v[0] for b, v in cbs.CBS_AGE_2022.items()}
cbs_cum = cal_.cbs_cumulative_expected(rates22)
ages = np.arange(0, 21)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(ages, cbs_cum[:21], "k-", lw=3, label="Israel: sum of CBS-2022 yearly rates (all private cars)")
for name, df in MARKETS.items():
    med = df[df.age.between(0, 20)].groupby("age").km.median()
    ax.plot(med.index, med.values, marker="o", ms=3, label=f"{name}: median odometer of listings")
ax.set_xlabel("age (years)"); ax.set_ylabel("km"); ax.grid(alpha=.3); ax.legend(fontsize=8); ax.set_title("Expected odometer by age"); plt.show()
print({a: round(float(cbs_cum[a])) for a in (3, 5, 7, 10, 15)})

{3: 78400, 5: 110400, 7: 140600, 10: 181600, 15: 242400}


## 3. The uncalibrated model against the official marginals

In [4]:
P = pd.concat([clean["d7_uk"], clean["d8_in"], clean["d3_cardekho"]], ignore_index=True)
train_A, _ = train_test_split(P, test_size=0.2, random_state=42, stratify=P["source"])
mA = cal_.fit_monotone(train_A)             # age + fuel only: make has no measurable value and is unreliable for unseen Israeli makes
FUELS = ["petrol", "diesel", "hybrid", "electric", "lpg_cng"]
curves = cal_.cumulative_curves(mA, FUELS)
A_SUP = cal_.supported_max_age(train_A.age)
RATES_KM = np.array([rates22[cbs.age_to_band(a)] * 1000 for a in range(cal_.MAX_AGE + 1)])
print(f"Training support: every age <= {A_SUP} has >= 300 vehicles in 7+8+3; above that the learned curve is flat (no information).")
print("Learned yearly increment of the age curve (petrol), km:", {a: round(float(x)) for a, x in enumerate(cal_.increments(curves['petrol'])) if a in (1, 3, 6, 9, 12, 15, 18, 21, 25)})

il = normalise_registry(pd.read_csv(INTERIM_DIR / "gov_registry_sample.csv"), reference_year=2026)
il = il[il.age >= 1].reset_index(drop=True)
il["own_cbs"] = il.ownership.map(cbs.REGISTRY_OWNERSHIP_TO_CBS)
il["fuel_cbs"] = il.fuel.map(cbs.REGISTRY_FUEL_TO_CBS)
il["band"] = il.age.map(cbs.age_to_band)
base_ml = cal_.implied_annual(curves, il.fuel, il.age)          # purely learned (what the model alone implies)
base = cal_.implied_annual(curves, il.fuel, il.age, A_SUP, RATES_KM)   # support-aware hybrid: learned where supported, CBS rate above
il["base"] = base
print(f"{len(il):,} registry vehicles (age >= 1). Registry shares: ownership {il.own_cbs.value_counts(normalize=True).round(3).to_dict()}")
T = {"age": rates22,
     "own": {o: cbs.CBS_OWNERSHIP_2022[o]["all"] for o in ["private", "leasing", "company", "rental"]},
     "fuel": cbs.CBS_FUEL_2022}
G = {"age": il.band.to_numpy(), "own": il.own_cbs.to_numpy(), "fuel": il.fuel_cbs.to_numpy(object)}

def marginals(values):
    rows = []
    for fam in ["age", "own", "fuel"]:
        for cat, tgt in T[fam].items():
            m = G[fam] == cat
            rows.append({"family": fam, "category": cat, "n": int(m.sum()), "model implied (thousand km/yr)": values[m].mean() / 1000, "CBS 2022": tgt})
    d = pd.DataFrame(rows)
    d["model/CBS"] = d["model implied (thousand km/yr)"] / d["CBS 2022"]
    return d
m0 = marginals(base_ml)
display(m0)
print(f"Fleet mean implied by the LEARNED model alone: {base_ml.mean()/1000:.1f} vs CBS 15.2 (2022) / 14.7 (2023-24)")
print(f"Share of Israeli registry vehicles older than the supported age ({A_SUP}): {(il.age > A_SUP).mean():.1%}  -> there the learned model alone has no information")

Training support: every age <= 13 has >= 300 vehicles in 7+8+3; above that the learned curve is flat (no information).
Learned yearly increment of the age curve (petrol), km: {1: 7626, 3: 9414, 6: 7624, 9: 4244, 12: 3521, 15: 9541, 18: 11053, 21: 0, 25: 0}


17,507 registry vehicles (age >= 1). Registry shares: ownership {'private': 0.892, 'leasing': 0.057, 'company': 0.039, 'rental': 0.011}


,family,category,n,model implied (thousand km/yr),CBS 2022,model/CBS
0,age,0-3,4116,10.320,19.600,0.527
1,age,4-6,3358,8.565,16.000,0.535
2,age,7-9,3085,4.078,14.200,0.287
3,age,10-13,3236,6.769,12.600,0.537
4,age,14-19,3138,4.861,11.500,0.423
5,age,20-29,574,0.000,9.300,0.000
6,age,30+,0,NaN,4.600,NaN
7,own,private,15618,6.586,13.900,0.474
8,own,leasing,1003,10.403,28.300,0.368
9,own,company,685,8.329,24.400,0.341


Fleet mean implied by the LEARNED model alone: 6.9 vs CBS 15.2 (2022) / 14.7 (2023-24)
Share of Israeli registry vehicles older than the supported age (13): 21.2%  -> there the learned model alone has no information


The model *under*-estimates every group, but **not by a constant factor**: the age profile is wrong in shape (the 7+8+3 listings contain few old cars and trees cannot extrapolate), and the model has no notion of ownership,
which in Israel separates 14k from 28k km/yr. A single global scalar therefore cannot fix it.

## 4. How the calibration works, and does it hold out of sample?
Take the model's yearly increments `inc(a, fuel)`. Multiply them by factors `f_age[band(a)] · f_own[ownership] · f_fuel[fuel]`, chosen by **iterative proportional fitting on means** so that the implied means on the real Israeli registry sample
equal the official CBS means for the families used. The calibrated odometer is the cumulative sum of the calibrated increments.
The test of honesty: calibrate on some CBS families and check the **others** (never fitted) and the **joint cells** of Table 8.

In [5]:
def run_cfg(families):
    factors, mult = cal_.rake_means(base / 1000, G, T, families)     # CBS targets are in thousand km
    return factors, base * mult

CFG = {"C0 none": [], "C1 scalar (fleet 15.2)": None, "C2 age": ["age"], "C3 age+ownership": ["age", "own"], "C4 age+ownership+fuel": ["age", "own", "fuel"],
       "L1 ownership+fuel (age held out)": ["own", "fuel"], "L2 age+fuel (ownership held out)": ["age", "fuel"]}
res, vals = {}, {}
for name, fams in CFG.items():
    if name == "C0 none":
        v = base
    elif fams is None:
        v = base * (15.2 * 1000 / base.mean())
    else:
        v = run_cfg(fams)[1]
    vals[name] = v
    d = marginals(v); d["abs % error"] = (d["model/CBS"] - 1).abs() * 100
    res[name] = d
summary = []
for name, fams in CFG.items():
    d = res[name]; used = set(fams or [])
    row = {"calibration": name}
    for fam in ["age", "own", "fuel"]:
        e = d[d.family == fam]["abs % error"]
        row[f"{fam}: mean |err| %"] = e.mean()
        row[f"{fam} used?"] = "fitted" if fam in used else "HELD OUT"
    summary.append(row)
display(pd.DataFrame(summary).set_index("calibration"))

,age: mean |err| %,age used?,own: mean |err| %,own used?,fuel: mean |err| %,fuel used?
calibration,,,,,,
C0 none,35.229,HELD OUT,57.631,HELD OUT,50.560,HELD OUT
C1 scalar (fleet 15.2),36.761,HELD OUT,27.011,HELD OUT,13.387,HELD OUT
C2 age,0.000,fitted,23.354,HELD OUT,13.856,HELD OUT
C3 age+ownership,2.699,fitted,0.000,fitted,13.591,HELD OUT
C4 age+ownership+fuel,2.699,fitted,0.000,fitted,0.000,fitted
L1 ownership+fuel (age held out),32.894,HELD OUT,0.000,fitted,0.000,fitted
L2 age+fuel (ownership held out),0.000,fitted,21.999,HELD OUT,0.000,fitted


In [6]:
# Joint cells of Table 8 (new = ages 1-3, old = ages >=4) x ownership: never used in any fit
rows = []
new_mask = (il.age <= 3).to_numpy()
for name in ["C0 none", "C2 age", "C3 age+ownership", "C4 age+ownership+fuel"]:
    v = vals[name]
    for o in ["private", "leasing", "company", "rental"]:
        for grp, mask in [("new", new_mask), ("old", ~new_mask)]:
            m = (G["own"] == o) & mask
            if m.sum() >= 15:
                tgt = cbs.CBS_OWNERSHIP_2022[o][grp]
                rows.append({"calibration": name, "ownership": o, "age group": grp, "n": int(m.sum()), "model": v[m].mean() / 1000, "CBS": tgt, "abs % err": abs(v[m].mean() / 1000 / tgt - 1) * 100})
cells_df = pd.DataFrame(rows)
display(cells_df.pivot_table(index=["ownership", "age group"], columns="calibration", values="abs % err").round(1))
print("mean abs % error over the 8 joint cells:", cells_df.groupby("calibration")["abs % err"].mean().round(1).to_dict())

calibration          C0 none  C2 age  C3 age+ownership  C4 age+ownership+fuel
ownership age group                                                          
company   new         62.300  28.400             5.500                  6.900
          old         65.900  32.600             6.800                  8.400
leasing   new         63.000  29.700             0.600                  0.900
          old         66.600  33.500            11.600                 16.600
private   new         35.500  22.400             1.700                  2.300
          old         41.700   0.300             0.100                  0.100
rental    new         62.200  28.300             1.000                  1.400
          old         63.800  34.200             7.600                 12.000

mean abs % error over the 8 joint cells: {'C0 none': 57.6, 'C2 age': 26.2, 'C3 age+ownership': 4.4, 'C4 age+ownership+fuel': 6.1}


**Reading.** (i) Calibrating on *age* alone leaves ownership/fuel marginals far off; adding ownership fixes ownership **and** pulls the held-out fuel and joint cells closer — i.e. the multiplicative structure is roughly right for Israel.
(ii) When *age* is held out (L1) the age profile is badly reproduced: the learned shape cannot be recovered from ownership/fuel means, so the age factors carry the real information.
(iii) The joint cells (never fitted) show how much interaction (e.g. leasing cars drive more *when new*) a purely multiplicative model misses.

In [7]:
f3, _ = run_cfg(["age", "own"])
f4, _ = run_cfg(["age", "own", "fuel"])
fac = pd.concat({k: pd.Series(v) for k, v in f3.items()}, axis=0).rename("C3 factor (age+ownership)").to_frame()
fac["C4 factor (+fuel)"] = pd.concat({k: pd.Series(v) for k, v in f4.items()}, axis=0)
display(fac)
print("age factors > 1 mean the learned model under-predicts that age band; they absorb both the lower-usage regime of the listings and the missing shape for old cars.")

C3 factor (age+ownership)  C4 factor (+fuel)
age 0-3                          1.666              2.077
    4-6                          1.939              2.564
    7-9                          3.689              4.917
    10-13                        1.995              2.694
    14-19                        1.076              1.463
    20-29                        1.082              1.480
    30+                          1.082              1.480
own private                      0.947              0.943
    leasing                      1.612              1.707
    company                      1.506              1.499
    rental                       1.574              1.695

age factors > 1 mean the learned model under-predicts that age band; they absorb both the lower-usage regime of the listings and the missing shape for old cars.


### 4.1 Is the *learned fuel effect* worth keeping? (ablation)
The fuel curves come from UK/India listings (e.g. diesel listings drive more there). Stacked on top of the Israeli ownership/age factors they can double-count.
Variant: use the **petrol curve for every fuel** and calibrate on age + ownership only; compare held-out fuel marginals and the 8 joint cells.

In [8]:
curves_pet = {f: curves["petrol"] for f in FUELS}
base_pet = cal_.implied_annual(curves_pet, il.fuel, il.age, A_SUP, RATES_KM)
_, mult_pet = cal_.rake_means(base_pet / 1000, G, T, ["age", "own"])
v_pet = base_pet * mult_pet

def cell_error(v):
    errs = []
    for o in ["private", "leasing", "company", "rental"]:
        for grp, mask in [("new", new_mask), ("old", ~new_mask)]:
            m = (G["own"] == o) & mask
            if m.sum() >= 15:
                errs.append(abs(v[m].mean() / 1000 / cbs.CBS_OWNERSHIP_2022[o][grp] - 1) * 100)
    return float(np.mean(errs))

rows = []
for name, v in [("C3, learned fuel curves", vals["C3 age+ownership"]), ("C3, petrol curve for every fuel", v_pet)]:
    f = marginals(v); f = f[f.family == "fuel"]
    row = {"variant": name, "joint-cell mean |err| %": cell_error(v), "held-out fuel marginals mean |err| %": float(((f["model/CBS"] - 1).abs() * 100).mean())}
    for _, r in f.iterrows():
        row[f"{r['category']} (CBS {r['CBS 2022']})"] = r["model implied (thousand km/yr)"]
    rows.append(row)
display(pd.DataFrame(rows))
ex = []
for fuel, own in [("diesel", "company"), ("hybrid", "private"), ("petrol", "private")]:
    ex.append({"vehicle": f"{fuel}, {own}, age 12",
               "C3 learned fuel curves (km)": float(cal_.calibrated_odometer(curves, f3, [fuel], [12], [own], None, A_SUP, RATES_KM)[0]) if "f3" in globals() else np.nan})

,variant,joint-cell mean |err| %,held-out fuel marginals mean |err| %,gasoline (CBS 14.0),diesel (CBS 22.8),hybrid (CBS 21.1),gas_lpg (CBS 18.8)
0,"C3, learned fuel curves",4.362,13.591,14.528,20.550,21.769,11.740
1,"C3, petrol curve for every fuel",3.973,23.568,15.040,15.106,17.608,11.928


## 5. Does the *procedure* improve individual-level error? Tested where individual ground truth exists
Israel has no open per-vehicle odometer data, so individual accuracy cannot be measured there. What can be tested is the **method**: train on market X, calibrate with **aggregates only**
(mean odometer per age band of market Y — the analogue of an official table), and evaluate on **individual vehicles of market Y** never used for calibration.
Baselines: uncalibrated; scalar; band calibration; *lookup* (band mean of Y, no model); and an *oracle* trained on Y's individual rows.
This is optimistic for Israel (here the aggregates are cumulative odometer means; for Israel they must be inferred from yearly rates — section 2).

In [9]:
def fit_q(train, q):
    pipe = mx.build_pipeline(numeric=["age"], low_card=["fuel"], make_enc=None, algo="xgb")
    pipe.set_params(est=XGBRegressor(objective="reg:quantileerror", quantile_alpha=q, n_estimators=300, learning_rate=0.05, max_depth=4,
                                     subsample=0.8, tree_method="hist", n_jobs=mx.N_JOBS, random_state=0))
    return pipe.fit(train, train.km)

src_models = {n: cal_.fit_monotone(df) for n, df in MARKETS.items()}
src_q = {n: (fit_q(df, 0.1), fit_q(df, 0.9)) for n, df in MARKETS.items()}
BANDS = cbs.BAND_LABELS
bandof = lambda s: s.map(cbs.age_to_band).to_numpy()

def metrics(y, p):
    m = mx.regression_metrics(y, p); m["bias"] = float(np.mean(np.clip(p, 0, None) - y)); return m

rows, cov_rows = [], []
for src in MARKETS:
    for tgt in MARKETS:
        if src == tgt:
            continue
        calib, test = train_test_split(MARKETS[tgt], test_size=0.5, random_state=0)
        mdl = src_models[src]
        p_cal, p_te = np.clip(mdl.predict(calib), 0, None), np.clip(mdl.predict(test), 0, None)
        b_cal, b_te = bandof(calib.age), bandof(test.age)
        s = calib.km.mean() / p_cal.mean()
        fb = {b: (calib.km[b_cal == b].mean() / p_cal[b_cal == b].mean() if (b_cal == b).sum() >= 30 else s) for b in BANDS}
        lk = {b: (calib.km[b_cal == b].mean() if (b_cal == b).sum() >= 30 else calib.km.mean()) for b in BANDS}
        preds = {"uncalibrated": p_te, "scalar": p_te * s, "age-band": p_te * np.array([fb[b] for b in b_te]),
                 "lookup (band mean only)": np.array([lk[b] for b in b_te]),
                 "oracle (trained on target individuals)": np.clip(cal_.fit_monotone(calib).predict(test), 0, None)}
        for meth, p in preds.items():
            rows.append({"train on": src, "calibrate+test on": tgt, "method": meth, **metrics(test.km.to_numpy(), p)})
        lo, hi = src_q[src][0].predict(test), src_q[src][1].predict(test)
        lo, hi = np.minimum(lo, hi), np.maximum(lo, hi)
        k = np.array([fb[b] for b in b_te])
        cov_rows.append({"train on": src, "test on": tgt,
                         "80% interval coverage (uncalibrated)": float(((test.km >= lo) & (test.km <= hi)).mean()),
                         "coverage after scaling interval by band factors": float(((test.km >= lo * k) & (test.km <= hi * k)).mean()),
                         "median width after scaling (km)": float(np.median((hi - lo) * k))})
sim = pd.DataFrame(rows)
display(sim.pivot_table(index=["train on", "calibrate+test on"], columns="method", values="RMSE")[["uncalibrated", "scalar", "age-band", "lookup (band mean only)", "oracle (trained on target individuals)"]])
display(sim.pivot_table(index=["train on", "calibrate+test on"], columns="method", values="R2")[["uncalibrated", "scalar", "age-band", "lookup (band mean only)", "oracle (trained on target individuals)"]])
print("Mean over the 6 market pairs:"); display(sim.groupby("method")[["RMSE", "MAE", "R2", "bias"]].mean().loc[["uncalibrated", "scalar", "age-band", "lookup (band mean only)", "oracle (trained on target individuals)"]])
print("Prediction-interval check:"); display(pd.DataFrame(cov_rows))

method                      uncalibrated     scalar   age-band  lookup (band mean only)  oracle (trained on target individuals)
train on calibrate+test on                                                                                                     
India    UK                   23,635.329 22,961.282 22,055.313               25,172.926                              21,036.948
         US                  140,915.775 78,673.389 76,068.886               74,191.625                              72,989.446
UK       India                38,533.003 30,228.968 28,997.874               31,324.917                              28,525.941
         US                  111,749.755 76,751.497 74,548.334               74,191.625                              72,989.446
US       India                98,384.229 29,548.589 29,555.784               31,324.917                              28,525.941
         UK                   57,858.987 22,170.031 21,601.762               25,172.926                              21,036.948

method                      uncalibrated  scalar  age-band  lookup (band mean only)  oracle (trained on target individuals)
train on calibrate+test on                                                                                                 
India    UK                        0.508   0.536     0.572                    0.442                                   0.611
         US                       -1.305   0.282     0.328                    0.361                                   0.382
UK       India                    -0.159   0.287     0.344                    0.234                                   0.365
         US                       -0.450   0.316     0.355                    0.361                                   0.382
US       India                    -6.556   0.318     0.318                    0.234                                   0.365
         UK                       -1.946   0.567     0.589                    0.442                                   0.611

Mean over the 6 market pairs:


,RMSE,MAE,R2,bias
method,,,,
uncalibrated,"78,512.846","65,419.244",-1.651,"-7,415.665"
scalar,"43,388.959","32,043.821",0.384,93.420
age-band,"42,137.992","31,124.892",0.418,85.620
lookup (band mean only),"43,563.156","32,705.616",0.346,75.237
oracle (trained on target individuals),"40,850.778","30,080.784",0.452,38.340


Prediction-interval check:


,train on,test on,80% interval coverage (uncalibrated),coverage after scaling interval by band factors,median width after scaling (km)
0,UK,India,0.680,0.735,"53,332.119"
1,UK,US,0.268,0.753,"163,648.790"
2,India,UK,0.797,0.821,"45,603.712"
3,India,US,0.123,0.766,"176,487.431"
4,US,UK,0.616,0.869,"55,010.497"
5,US,India,0.343,0.808,"66,114.245"


**What this experiment shows (and does not).**
* Aggregate-only calibration removes most of the *bias* (systematic level error) and cuts individual-level RMSE substantially compared with the uncalibrated cross-market model.
* It does **not** make individual predictions precise: even the oracle (trained on the target market's individual vehicles) keeps R² well below 1, and the calibrated model cannot beat what the individual data of that market allow.
* **Intervals:** the table above compares the nominal-80% interval before and after scaling it by the band factors. Whatever it shows here, the spread of individual vehicles around the population mean is a separate quantity that official *means* do not contain: in Israel it can only be checked with individual-level odometer data.

## 6. What a calibrated estimate would look like (illustrative, population level)

In [10]:
f3, _ = run_cfg(["age", "own"])          # recommended: age + ownership (fuel factors are confounded with ownership and did not help on held-out cells)
def odo(fuel, a, own):
    return float(cal_.calibrated_odometer(curves, f3, [fuel], [a], [own], None, A_SUP, RATES_KM)[0])
rows = []
for fuel, own in [("petrol", "private"), ("petrol", "leasing"), ("diesel", "company"), ("hybrid", "private")]:
    for a in (3, 7, 12):
        rows.append({"vehicle": f"{fuel}, {own}", "age": a, "uncalibrated learned model (km)": float(curves[fuel][a]), "calibrated C3 (km)": odo(fuel, a, own),
                     "CBS-rates-only, all private cars (km)": float(cbs_cum[a])})
display(pd.DataFrame(rows))
print("These are estimated POPULATION-LEVEL expectations for a vehicle class. The individual 80% range in the training markets was ~ +/-24,000 km (notebook 01); nothing here narrows it, and for Israel it is unknown.")

,vehicle,age,uncalibrated learned model (km),calibrated C3 (km),"CBS-rates-only, all private cars (km)"
0,"petrol, private",3,"33,406.426","52,719.129","78,400.000"
1,"petrol, private",7,"64,868.637","121,946.773","140,600.000"
2,"petrol, private",12,"88,351.102","173,242.890","206,800.000"
3,"petrol, leasing",3,"33,406.426","89,743.392","78,400.000"
4,"petrol, leasing",7,"64,868.637","207,589.109","140,600.000"
5,"petrol, leasing",12,"88,351.102","294,910.118","206,800.000"
6,"diesel, company",3,"47,241.656","118,546.950","78,400.000"
7,"diesel, company",7,"86,830.312","251,396.310","140,600.000"
8,"diesel, company",12,"134,501.469","421,321.289","206,800.000"
9,"hybrid, private",3,"42,041.738","66,346.632","78,400.000"


These are estimated POPULATION-LEVEL expectations for a vehicle class. The individual 80% range in the training markets was ~ +/-24,000 km (notebook 01); nothing here narrows it, and for Israel it is unknown.


## 7. Conclusions (population level only)
1. **Official data confirmed at source.** CBS Publication 1923 (odometer readings from the annual inspection, whole fleet): private cars 15.2 thousand km/yr (2022), 14.7 (2023-24, CBS release 2025);
   by age 19.6 → 4.6; by ownership private 13.9 / leasing 28.3 / company 24.4 / rental 26.1; no electric-vehicle data yet. My transcription reproduces the published totals.
2. **The learned model alone is far from Israel:** it implies ~45% of the official level, with no information above age 13 (21% of the Israeli fleet) and no notion of ownership, the strongest driver in the CBS tables.
3. **Calibration works as a method (held-out evidence):** calibrating on age + ownership reduces the mean error on the 8 Table-8 joint cells (never fitted) from 57.6% to **4.4%** (max 11.6%); adding fuel factors does **not** help (6.1%) because CBS fuel means are confounded with ownership/age.
   Ownership information is independent and necessary (holding it out leaves 22% error); age information is necessary (holding it out leaves 33%).
4. **Method test with individual ground truth (market transfer):** aggregate-only calibration cuts individual RMSE from ~78.5k to ~42.1k km (oracle trained on the target's individual vehicles: ~40.9k) and removes the bias; R² ≈ 0.42 vs 0.45.
   Scaling the interval by the band factors brought nominal-80% coverage into the 74–87% range in this simulation.
5. **What this does NOT show.** No individual-level accuracy in Israel (no open per-vehicle odometer data; the CBS has it but publishes only aggregates). Scale factors rely on assumptions: stationary yearly rates across cohorts, first-year partial use, survivorship, 2022 detail combined with the 2023-24 fleet mean (-3%), and the simulation used cumulative means whereas Israel's must be inferred from yearly rates.
   Electric vehicles (6.8% of the fleet) are not covered by any official figure and were treated like petrol cars (not validated). Stacked effects (e.g. diesel × company) are extrapolations of two separate factors.
6. **Marginal value of the learned regression.** Against a pure official lookup (age-band means only) the calibrated model is only ~3% better in RMSE in the simulation: the learned part contributes mostly the within-support age shape and fuel curve; the *level* of Israeli usage comes from the official statistics. This should be stated openly.